# Visualizing Climate-Risk Trends for Brazilian State Capitals

This script loads the compiled annual metrics and trend statistics generated by the climate-risk screening pipeline and uses `aidviz` to create scientific storytelling visualizations for three key capitals:
1. **Brasília** (represented as the peak desertification/drying pressure baseline)
2. **Manaus** (represented as the peak flooding/deluge pressure baseline)
3. **São Paulo** (represented as a transitional baseline)

Output plots are saved to `outputs/brazil_capitals/`.


In [1]:
import logging
import warnings
from pathlib import Path
import pandas as pd

from aidweather import get_config

# Configure logging using built-in configuration settings
log_cfg = get_config().logging_config()
log_handlers = [logging.StreamHandler()]  # Console logger is always active

if log_cfg.get("enabled", False):
    log_file = log_cfg.get("filename")
    if log_file:
        log_file_path = Path(log_file)
        log_file_path.parent.mkdir(parents=True, exist_ok=True)
        log_handlers.append(logging.FileHandler(log_file_path))

logging.basicConfig(
    level=getattr(logging, log_cfg.get("level", "INFO")),
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=log_handlers,
    force=True,
)
logger = logging.getLogger("brazil_capitals_plots")


In [2]:
# Define directory paths
if "__file__" in globals():
    script_dir = Path(__file__).resolve().parent
    IS_MAIN = __name__ == "__main__"
else:
    script_dir = Path.cwd()
    IS_MAIN = True

# We assume the output files exist in outputs/brazil_capitals/
# Let's locate the output directory
output_dir = script_dir.parent.parent / "outputs" / "brazil_capitals"

annual_path = output_dir / "annual_metrics.parquet"
trends_path = output_dir / "trends.csv"


In [3]:
if IS_MAIN:
    if not annual_path.exists() or not trends_path.exists():
        logger.error(
            f"Required datasets not found in {output_dir}. "
            "Please run examples/brazil/brazil_capitals_climate_risk.py first "
            "to generate the annual metrics and trend datasets."
        )
    else:
        logger.info(f"Loading annual metrics from {annual_path}...")
        annual_df = pd.read_parquet(annual_path)
        
        logger.info(f"Loading trend statistics from {trends_path}...")
        trends_df = pd.read_csv(trends_path)
        
        # Import ClimateStoryFigure from aidviz
        try:
            from aidviz import ClimateStoryFigure
            logger.info("Imported ClimateStoryFigure from aidviz.")
        except ImportError as e:
            logger.critical(f"Failed to import ClimateStoryFigure from aidviz: {e}")
            raise e
            
        # Initialize the figure composition layer
        story = ClimateStoryFigure(annual_df=annual_df, trends_df=trends_df)
        
        capitals_to_plot = ["Brasília", "Manaus", "São Paulo"]
        
        for capital in capitals_to_plot:
            logger.info(f"Composing climate story visualization for {capital}...")
            
            try:
                # Plot the individual capital's timeline and trends
                fig = story.plot_entity(capital, entity_col="capital")
                
                # Define export paths
                html_path = output_dir / f"{capital.lower().replace(' ', '_')}_climate_story.html"
                png_path = output_dir / f"{capital.lower().replace(' ', '_')}_climate_story.png"
                
                # Save as interactive HTML report
                logger.info(f"Saving interactive plot to {html_path}...")
                story.save_html(fig, str(html_path))
                
                # Optionally attempt to export static PNG (depends on kaleido availability)
                try:
                    import kaleido
                    logger.info(f"Saving static PNG to {png_path}...")
                    fig.write_image(str(png_path), scale=2)
                except ImportError:
                    logger.info(
                        "kaleido is not installed. Skipping static PNG export. "
                        "Interactive HTML is still generated successfully."
                    )
            except Exception as ex:
                logger.error(f"Failed to generate plot for {capital}: {ex}", exc_info=True)
                
        logger.info("Climate risk visualization pipeline complete.")


2026-07-01 17:24:07,700 [INFO] brazil_capitals_plots: Loading annual metrics from /home/clever/aidbio/dev/aidweather/outputs/brazil_capitals/annual_metrics.parquet...
2026-07-01 17:24:07,715 [INFO] brazil_capitals_plots: Loading trend statistics from /home/clever/aidbio/dev/aidweather/outputs/brazil_capitals/trends.csv...
2026-07-01 17:24:08,220 [INFO] brazil_capitals_plots: Imported ClimateStoryFigure from aidviz.
2026-07-01 17:24:08,221 [INFO] brazil_capitals_plots: Composing climate story visualization for Brasília...
2026-07-01 17:24:08,385 [INFO] brazil_capitals_plots: Saving interactive plot to /home/clever/aidbio/dev/aidweather/outputs/brazil_capitals/brasília_climate_story.html...
2026-07-01 17:24:08,410 [INFO] brazil_capitals_plots: kaleido is not installed. Skipping static PNG export. Interactive HTML is still generated successfully.
2026-07-01 17:24:08,411 [INFO] brazil_capitals_plots: Composing climate story visualization for Manaus...
2026-07-01 17:24:08,456 [INFO] brazil_